# Deploy a Custom LLM (Qwen2.5-7B-Instruct) on Databricks Model Serving with vLLM

This notebook downloads a HuggingFace LLM, validates it locally in a **Serverless GPU notebook**, then deploys it to **Databricks Model Serving** via the new Serverless Optimized Deployments path backed by vLLM. See [Serve custom LLMs](https://docs.databricks.com/aws/en/machine-learning/model-serving/serve-custom-llms).

**Compute** — Serverless GPU notebook (Databricks AI Runtime), **A10 (24 GB)**.

**Model** — `Qwen/Qwen2.5-7B-Instruct`. Ungated, fits an A10 at `dtype=float16` (~14 GB weights + KV cache). Swap `MODEL_REPO_ID` for any vLLM-compatible model that fits the GPU.

Edit the **Configuration** cell before running the rest of the notebook.

## Set up the environment (Serverless GPU with A10)

In [0]:
%sh
nvidia-smi

Tue Jun  2 20:09:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   31C    P8             26W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [0]:
# Serving runtime requirements. Pinned versions match the Databricks starter.
%pip install vllm==0.11.2 transformers==4.57.6 openai==2.17.0 opencv-python-headless==4.12.* mlflow==3.12.0 hf_transfer==0.1.9
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# /Workspace can't hold multi-GB weights — work from local disk.
import os, tempfile
workdir = tempfile.mkdtemp()
os.chdir(workdir)
print("workdir:", workdir)

workdir: /tmp/tmpbrk8gs72


## Configuration

All knobs in one place. The block below is split into:
1. **Model + paths** — what to download, where it lands.
2. **vLLM tuning** — required + optional flags (set the optional ones to `None`/`False` to drop them).
3. **Ports** — local test must use 3000–3999 on Serverless GPU; Serving always uses 8080.
4. **Unity Catalog + Endpoint** — where the registered model + endpoint go.

In [0]:
from databricks.sdk.service.serving import ServingModelWorkloadType

# --- 1) Model + paths -----------------------------------------------------
MODEL_REPO_ID = "Qwen/Qwen2.5-7B-Instruct"
ARTIFACTS_PATH = "qwen25_7b"   # Doubles as the local dir AND the MLflow artifact key — vLLM resolves it correctly in both contexts.
SERVED_MODEL_NAME = "qwen"     # Value clients pass in `model` when hitting vLLM directly.

# --- 2) vLLM tuning -------------------------------------------------------
# Required defaults.
DTYPE = "float16"                  # float16 | bfloat16 | float32 — A10 has no bf16 tensor cores, fp16 is best.
MAX_MODEL_LEN = 16384              # Context window. Larger = more KV cache memory.
GPU_MEMORY_UTILIZATION = 0.85      # Fraction of GPU memory vLLM may use.

# Optional tunables — set to None / False to omit the flag entirely.
ENFORCE_EAGER = False              # True: skip CUDA graph capture. Faster cold start, ~10-20% lower throughput.
TENSOR_PARALLEL_SIZE = 1           # >1 only if the endpoint has multiple GPUs (A10 is single-GPU, leave at 1).
MAX_NUM_SEQS = None                # e.g. 64. Cap on concurrent requests; raises throughput, costs KV memory.
MAX_NUM_BATCHED_TOKENS = None      # e.g. 8192. Throughput vs latency knob for prefill batching.
KV_CACHE_DTYPE = None              # e.g. "fp8". Halves KV cache memory at minor quality cost.
QUANTIZATION = None                # e.g. "fp8" | "awq" | "gptq". Required if the HF repo is pre-quantized.
SWAP_SPACE = None                  # GiB of CPU RAM for KV cache spill (e.g. 4).
EXTRA_VLLM_ARGS = []               # Escape hatch: any other flags, e.g. ["--disable-log-requests"].

# --- 3) Ports -------------------------------------------------------------
LOCAL_PORT = 3080      # Serverless GPU only allows inbound on 3000-3999.
SERVING_PORT = 8080    # Required by Model Serving — do not change.

# --- 4) Unity Catalog + Endpoint -----------------------------------------
UC_MODEL_NAME = "agents.custom_llm.qwen25_7b_instruct"   # catalog.schema.model — must be writable by you.

ENDPOINT_NAME = "qwen25-7b-endpoint"                       # Unique within the workspace.
WORKLOAD_TYPE = ServingModelWorkloadType.GPU_MEDIUM        # T4: GPU_SMALL · A10: GPU_MEDIUM · H100: GPU_XLARGE
WORKLOAD_SIZE = "Small"                                    # Small | Medium | Large
SCALE_TO_ZERO_ENABLED = False                              # Entrypoint models don't support autoscaling; workload_size sets the fixed concurrency.

## Download the model

In [0]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_REPO_ID,
    local_dir=ARTIFACTS_PATH,
)

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

'/tmp/tmpbrk8gs72/qwen25_7b'

## Test the model in the notebook

The same `entrypoint(port)` string is used twice:
- Locally with `port=LOCAL_PORT` (3080) for validation.
- Stored in MLflow metadata with `port=SERVING_PORT` (8080) so Model Serving knows how to launch vLLM on the fleet.

Because `ARTIFACTS_PATH` is used both as the local directory name **and** as the MLflow artifact key, the literal `--model qwen25_7b` resolves correctly in both contexts.

In [0]:
def entrypoint(port: int) -> str:
    args = [
        "python", "-u", "-m", "vllm.entrypoints.openai.api_server",
        "--model", ARTIFACTS_PATH,
        "--served-model-name", SERVED_MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(port),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    ]
    if ENFORCE_EAGER:
        args.append("--enforce-eager")
    if MAX_NUM_SEQS is not None:
        args += ["--max-num-seqs", str(MAX_NUM_SEQS)]
    if MAX_NUM_BATCHED_TOKENS is not None:
        args += ["--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS)]
    if KV_CACHE_DTYPE is not None:
        args += ["--kv-cache-dtype", KV_CACHE_DTYPE]
    if QUANTIZATION is not None:
        args += ["--quantization", QUANTIZATION]
    if SWAP_SPACE is not None:
        args += ["--swap-space", str(SWAP_SPACE)]
    args += EXTRA_VLLM_ARGS
    return " ".join(args)

print(entrypoint(LOCAL_PORT))

python -u -m vllm.entrypoints.openai.api_server --model qwen25_7b --served-model-name qwen --host 0.0.0.0 --port 3080 --dtype float16 --max-model-len 16384 --gpu-memory-utilization 0.85 --tensor-parallel-size 1


In [0]:
import subprocess

# Start vLLM in the background. Logs go to process.log so the %sh tail in the next cell can wait on readiness.
log = open("process.log", "w")
subprocess.Popen(
    ["bash", "-lc", entrypoint(LOCAL_PORT)],
    stdout=log,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

<Popen: returncode: None args: ['bash', '-lc', 'python -u -m vllm.entrypoint...>

In [0]:
%sh
# Tail logs until vLLM is ready. If this hangs, vLLM startup probably encountered an error — scroll up in process.log.
tail -f process.log | sed -u '/Application startup complete/q'

/local_disk0/.ephemeral_nfs/envs/pythonEnv-86c41ef5-05e7-4213-85ce-b13307f1f730/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO 06-02 20:10:21 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=2048.
(APIServer pid=11992) INFO 06-02 20:10:21 [api_server.py:1977] vLLM API server version 0.11.2
(APIServer pid=11992) INFO 06-02 20:10:21 [utils.py:253] non-default args: {'host': '0.0.0.0', 'port': 3080, 'model': 'qwen25_7b', 'dtype': 'float16', 'max_model_len': 16384, 'served_model_name': ['qwen'], 'gpu_memory_utilization': 0.85}
(APIServer pid=11992) INFO 06-02 20:10:21 [model.py:631] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=11992) WARNING 06-02 20:10:21 [model.py:1971] Casting torch.bfloat16 t

In [0]:
import requests

resp = requests.post(
    f"http://localhost:{LOCAL_PORT}/invocations",
    json={"messages": [{"role": "user", "content": "Give me three creative names for a coffee shop near a beach."}]},
)
print(resp.json()["choices"][0]["message"]["content"])

Certainly! Here are three creative names for a coffee shop located near the beach:

1. **Sands & Sips** - This name combines the idea of sand (representing the beach) with sips (referring to coffee), creating a playful and inviting atmosphere.

2. **Wave Brews** - This name evokes the sound of waves crashing on the shore while also hinting at the coffee being served, making it both descriptive and appealing.

3. **Seaside Roast** - This name directly references the location (seaside) and the core product (roast), suggesting a cozy and comforting environment where guests can enjoy their coffee overlooking the ocean.

Each of these names aims to capture the essence of a coastal setting while emphasizing the quality and experience of the coffee shop.


In [0]:
import requests, json

# vLLM streams completions as SSE: one JSON chunk per `data: ` line, terminated by `data: [DONE]`.
resp = requests.post(
    f"http://localhost:{LOCAL_PORT}/invocations",
    json={"messages": [{"role": "user", "content": "Tell me a story that is about 300 words!"}], "stream": True},
    stream=True,
)

for line in resp.iter_lines():
    if not line:
        continue
    if line == b"data: [DONE]":
        break
    if line.startswith(b"data: "):
        data = json.loads(line[6:])
        delta = data["choices"][0].get("delta", {})
        if "content" in delta:
            print(delta["content"], end="", flush=True)

In the heart of a vast, enchanted forest lived a curious young girl named Elara. Elara had a peculiar talent; she could communicate with the trees and listen to their whispers. One crisp autumn morning, as the leaves were turning shades of orange and gold, Elara heard a distant call from a towering oak tree. It was a plea for help.

The oak explained that a group of mischievous fairies had been stealing its acorns, leaving the tree weak and vulnerable. Elara knew she had to help. She packed a small bag with some seeds and fruits and set out into the forest.

As she wandered deeper, she encountered various woodland creatures who offered her guidance and assistance. A wise old owl taught her how to navigate through the forest without disturbing the delicate ecosystem. A friendly squirrel showed her where the fairies liked to gather.

Finally, Elara found the fairies in a clearing, surrounded by piles of stolen acorns. The fairies, led by a mischievous sprite named Flicker, argued over wh

In [0]:
%sh
# Free the GPU before logging the model.
pkill -f vllm.entrypoints.openai.api_server

## Log the model with our custom entrypoint

`LLMModel.predict` is a **required placeholder** — Serving runs the `entrypoint` string, not Python. The interesting bits:
- `artifacts={"model_dir": ARTIFACTS_PATH}` bundles the weights into the MLflow model. (Note: the actual flag passed to vLLM is `--model {ARTIFACTS_PATH}`, which resolves to the artifact directory at serve time because the key matches.)
- `metadata["task"] = "llm/v1/chat"` declares this as a chat LLM.
- `metadata["entrypoint"] = entrypoint(SERVING_PORT)` — note **SERVING_PORT (8080)**, not the local 3080.

In [0]:
import mlflow
from mlflow.pyfunc.model import ChatModel, ChatCompletionResponse

class LLMModel(ChatModel):
    def predict(self, context, messages, params):
        return ChatCompletionResponse.from_dict({"choices": []})

model_info = mlflow.pyfunc.log_model(
    name=SERVED_MODEL_NAME,
    python_model=LLMModel(),
    artifacts={
        "model_dir": ARTIFACTS_PATH,
    },
    metadata={
        "task": "llm/v1/chat",
        "entrypoint": entrypoint(SERVING_PORT),
    },
    extra_pip_requirements=[
        "mlflow==3.12.0",
    ],
)
model_info.model_uri

/home/spark-86c41ef5-05e7-4213-85ce-b1/.ipykernel/11654/command-6547708289359493-3675869982:10: FutureWarning: ``mlflow.pyfunc.model.ChatModel`` is deprecated since 3.0.0. This method will be removed in a future release. Use ``ResponsesAgent`` instead.
  python_model=LLMModel(),
🔗 View Logged Model at: https://dbc-b1357123-778f.cloud.databricks.com/ml/experiments/1182716603180279/models/m-6106be3f607c4e80899982613943df46?o=7474655512364318
2026/06/02 20:11:25 WARNING mlflow.pyfunc: Default values for temperature, n and stream in ChatParams will be removed in the next release. Specify them in the input example explicitly if needed.
2026/06/02 20:11:25 INFO mlflow.pyfunc: Predicting on input example to validate output


2026/06/02 20:11:30 WARNING mlflow.utils.databricks_utils: Missing required environment variable `SPARK_LOCAL_REMOTE` or `SPARK_REMOTE`. These are necessary to initialize the WorkspaceClient with serverless compute in a subprocess in Databricks for UC function execution. Setting the value to 'true'.


'models:/m-6106be3f607c4e80899982613943df46'

## Register the model to Unity Catalog

In [0]:
import mlflow

# env_pack is required. Custom LLM Serving depends on Serverless Optimized Deployments. The endpoint will not work without it.
# https://docs.databricks.com/aws/en/machine-learning/model-serving/serverless-optimized-deployments
model_version = mlflow.register_model(
    model_info.model_uri,
    UC_MODEL_NAME,
    env_pack="databricks_model_serving",
)

Registered model 'agents.custom_llm.qwen25_7b_instruct' already exists. Creating a new version of this model...
Packing environment for Databricks Model Serving with install_dependencies True...


Installing model requirements...
2026/06/02 20:19:21 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to fetch model version's source run (with ID ) from tracking server. The source run may be deleted or inaccessible to the current user. No run link will be recorded for the model version.
2026/06/02 20:19:21 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to get model version source run's workspace ID from request headers. No run link will be recorded for the model version


Uploading artifacts:   0%|          | 0/43 [00:00<?, ?it/s]

Uploading /tmp/tmp32kn1bye/model/artifacts/qwen25_7b/model-00001-of-00004.safetensors:   0%|          | 0.00/3…

Uploading /tmp/tmp32kn1bye/model/artifacts/qwen25_7b/model-00003-of-00004.safetensors:   0%|          | 0.00/3…

Uploading /tmp/tmp32kn1bye/model/artifacts/qwen25_7b/model-00002-of-00004.safetensors:   0%|          | 0.00/3…

Uploading /tmp/tmp32kn1bye/model/artifacts/qwen25_7b/model-00004-of-00004.safetensors:   0%|          | 0.00/3…

Uploading /tmp/tmp32kn1bye/model/_databricks/model_environment.tar:   0%|          | 0.00/9.20G [00:00<?, ?iB/…

Uploading /tmp/tmp32kn1bye/model/_databricks/model_version.tar:   0%|          | 0.00/14.2G [00:00<?, ?iB/s]

🔗 Created version '2' of model 'agents.custom_llm.qwen25_7b_instruct': https://dbc-b1357123-778f.cloud.databricks.com/explore/data/models/agents/custom_llm/qwen25_7b_instruct/version/2?o=7474655512364318
Staging model agents.custom_llm.qwen25_7b_instruct version 2 for Databricks Model Serving...


## Create an endpoint with the model

| GPU      | `workload_type`  | Memory |
|----------|------------------|--------|
| T4       | `GPU_SMALL`      | 16 GB  |
| **A10**  | **`GPU_MEDIUM`** | **24 GB** |
| H100     | `GPU_XLARGE`     | 80 GB  |

In [0]:
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=UC_MODEL_NAME,
            entity_version=str(model_version.version),
            workload_type=WORKLOAD_TYPE,
            workload_size=WORKLOAD_SIZE,
            scale_to_zero_enabled=SCALE_TO_ZERO_ENABLED,
        )
    ]
)

w = WorkspaceClient()

# Use create if the endpoint doesn't exist, otherwise update its config.
try:
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=config,
        timeout=timedelta(minutes=30),
    )
except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint '{ENDPOINT_NAME}' already exists — updating config...")
        w.serving_endpoints.update_config_and_wait(
            name=ENDPOINT_NAME,
            served_entities=config.served_entities,
            timeout=timedelta(minutes=30),
        )
    else:
        raise

Endpoint 'qwen25-7b-endpoint' already exists — updating config...


## Query the ready endpoint

In [0]:
# Databricks SDK.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    messages=[ChatMessage(role=ChatMessageRole.USER, content="Hi, what model are you?")],
)
print(resp.choices[0].message.content)

I am Qwen, a large language model created by Alibaba Cloud. How can I assist you today?


In [0]:
# OpenAI client.
from openai import OpenAI

DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

response = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[
        {"role": "user", "content": "Explain what vLLM does in 2 sentences."},
    ],
)
print(response.choices[0].message.content)

vLLM is a high-performance inference engine designed to accelerate large language models, enabling faster and more efficient processing of text through techniques like parallelism and optimized memory usage. It aims to reduce latency and increase throughput for applications that rely on large language models.


In [0]:
# OpenAI client — streaming.
stream = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[
        {"role": "user", "content": "Hello, tell me a 200 word story"},
    ],
    stream=True,
)

for event in stream:
    delta = event.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)

In the heart of an ancient forest, there lived a wise old owl named Orion. Every night, under the glow of the full moon, he would perch atop the tallest tree and share tales with the animals gathered below. One evening, a young fox named Finn approached Orion, his tail tucked nervously between his legs. "Old friend," Orion hooted gently, "what troubles you, my little fox?"

Finn hesitated, then confessed that he had stolen some berries from a farmer’s field to feed his hungry family. Orion listened intently before speaking. "Stealing is not always wrong, Finn," he said softly. "But consider this: what if everyone took what they needed without asking? The fields would soon be bare."

Inspired by Orion’s wisdom, Finn returned to the farmer’s field the next day, offering to work in exchange for food. The farmer, moved by Finn’s honesty, agreed. From that day forward, Finn worked diligently, earning his meals and teaching others the value of hard work and integrity.

Orion watched proudly 